# Independent check on open data: does ex vivo drug sensitivity predict remission in AML?

QPOP's evidence comes from one group, with data that cannot be downloaded. This notebook uses an **independent, fully open** dataset instead:
Malani et al., *Cancer Discovery* 2022 (FIMM, Helsinki), "Implementing a Functional Precision Medicine Tumor Board for Acute Myeloid Leukemia".
Data: Zenodo [10.5281/zenodo.7370747](https://doi.org/10.5281/zenodo.7370747), CC BY 4.0. Get it with `sh scripts/fetch_open_data.sh`.

**Question.** Patients whose first treatment was intensive induction (cytarabine + an anthracycline) and who had a drug screen at diagnosis:
does the lab sensitivity of their own cells to those drugs predict complete remission better than age or the standard genetic risk class (ELN 2017)?

**Limits, stated up front.** Single drugs, not QPOP combinations. The outcome is "best disease status" (best response ever recorded), not response to the first induction alone.
Codes 98 and 251 occur in the data but are not in the published code list, so those patients are excluded. sDSS = selective drug sensitivity score (higher = more sensitive than healthy cells), as the dataset authors recommend.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

D = "../ext/malani2022/Functional_Precision_Medicine_Tumor_Board_AML/"
clin = pd.read_excel(D + "File_1.1_Clinical_summary_186_Patients.xlsx")
samp = pd.read_excel(D + "File_0_Common_sample_annotation_252S.xlsx")
sdss = pd.read_excel(D + "File_3.1_Drug_response_sDSS_164S_17Healthy.xlsx").set_index("Drug_name")

samp["Disease status"] = samp["Disease status"].str.strip()
samp["pid"] = samp["Sample_ID"].str.extract(r"^(AML_\d+)", expand=False)
dx = samp[(samp["Disease status"] == "Diagnosis") & samp["Sample_ID"].isin(sdss.columns)].drop_duplicates("pid")
m = dx.merge(clin, left_on="pid", right_on="Patient_ID")
first = m["Treatments_pt"].fillna("").str.split(",").str[0]
m = m[first.str.contains(r"induction|7\+3|\bIA\b|\bDA\b", case=False) & first.str.contains(r"Cyt|AraC|IA|DA")].copy()
CR, NR = {1, 2, 3, 4, 5}, {6, 7, 8, 96}       # code list: File_1.2
m["remission"] = m["best_disease_status"].map(lambda v: 1 if v in CR else (0 if v in NR else np.nan))
m = m.dropna(subset=["remission"])
for drug in ("Cytarabine", "Idarubicin", "Daunorubicin"):
    m[drug] = [sdss.loc[drug, s] for s in m["Sample_ID"]]
m["Cyt + Ida mean"] = m[["Cytarabine", "Idarubicin"]].mean(axis=1)
Y = m["remission"].astype(int).to_numpy()
print(f"diagnosis samples with drug screen: {len(dx)}; intensive induction with known outcome: {len(m)} "
      f"(remission {Y.sum()}, none {len(Y) - Y.sum()}); everyone-remits baseline accuracy {Y.mean():.0%}")

diagnosis samples with drug screen: 79; intensive induction with known outcome: 42 (remission 34, none 8); everyone-remits baseline accuracy 81%


In [2]:
rng = np.random.default_rng(0)

def auc(x, y):
    p, n = x[y == 1], x[y == 0]
    return float(np.mean((p[:, None] > n[None]) + 0.5 * (p[:, None] == n[None])))

def boot_ci(x, y, B=2000):
    v = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            v.append(auc(x[i], y[i]))
    return np.percentile(v, [2.5, 97.5])

# sign: higher score should mean remission; risk class and age point the other way
PRED = {"Cytarabine": 1, "Idarubicin": 1, "Daunorubicin": 1, "Cyt + Ida mean": 1,
        "eln2017_risk_class_qp": -1, "Age at diagnosis": -1}
AUC = {}
for col, sign in PRED.items():
    x = sign * m[col].to_numpy(float); ok = ~np.isnan(x)
    AUC[col] = (auc(x[ok], Y[ok]), *boot_ci(x[ok], Y[ok]), int(ok.sum()))
    print(f"{col:22s} n={AUC[col][3]}  AUC {AUC[col][0]:.2f}  95% CI {AUC[col][1]:.2f}-{AUC[col][2]:.2f}")

Cytarabine             n=42  AUC 0.67  95% CI 0.41-0.91
Idarubicin             n=42  AUC 0.44  95% CI 0.18-0.71
Daunorubicin           n=42  AUC 0.44  95% CI 0.22-0.69
Cyt + Ida mean         n=42  AUC 0.55  95% CI 0.28-0.81
eln2017_risk_class_qp  n=42  AUC 0.70  95% CI 0.46-0.91


Age at diagnosis       n=42  AUC 0.74  95% CI 0.57-0.89


In [3]:
names = list(AUC)
fig = go.Figure(go.Scatter(x=[AUC[k][0] for k in names], y=names, mode="markers", marker=dict(size=10),
                           error_x=dict(type="data", symmetric=False,
                                        array=[AUC[k][2] - AUC[k][0] for k in names],
                                        arrayminus=[AUC[k][0] - AUC[k][1] for k in names])))
fig.add_vline(x=0.5, line_dash="dot", annotation_text="chance")
fig.add_vline(x=0.808, line_dash="dash", annotation_text="QPOP AML claim (29 runs)")
fig.update_layout(xaxis=dict(title="AUC for predicting remission (95% bootstrap CI)", range=[0, 1]),
                  height=380, title="Open AML data: lab sensitivity vs cheap clinical predictors")
fig

In [4]:
fig = go.Figure()
for drug in ("Cytarabine", "Idarubicin"):
    for yv, name in ((1, "remission"), (0, "no remission")):
        fig.add_trace(go.Box(y=m.loc[m["remission"] == yv, drug], name=f"{drug}: {name}", boxpoints="all", jitter=0.4))
fig.update_layout(yaxis_title="sDSS (higher = more sensitive)", height=420, showlegend=False,
                  title="Lab sensitivity of each patient's cells, by clinical outcome")
fig

## What this says

1. In this independent, open cohort, lab sensitivity to the drugs the patients actually received does not separate those who reached remission from those who did not:
   AUC 0.67 for cytarabine and below 0.5 for the anthracyclines, all with intervals that include 0.5.
2. Age alone (AUC 0.74) and the routine genetic risk class (0.70) do at least as well, at no cost.
3. With only 8 patients without remission, every interval is wide. The data cannot confirm or rule out a modest effect. That is the same small-sample problem as the QPOP AML paper (11 non-responders), but here anyone can check it.
4. This tests single drugs at diagnosis, not QPOP's combination ranking. It is a check of the shared premise, "lab response of the patient's cells predicts clinical response", not of QPOP itself.

The open files hold no response per treatment line, so outcomes of drug-screen-guided therapy (listed as "DSRT-based" in the treatment field) cannot be checked here.